# Introduction to Hypothesis Testing: Ham or Spam

A spam detector must decide, for every email, between two competing claims: it's **ham** (a normal email) or it's **spam**. The safe default is ham — deleting a good email hurts more than letting one spam through — and that default has a name: the **null hypothesis** $H_0$, the baseline where "nothing is happening." The claim you're trying to establish — spam — is the **alternative hypothesis** $H_1$. Evidence (sender, attachments, trigger phrases like *earn extra cash* and *act immediately*) can pile up against $H_0$ until you **reject** it; but the logic is deliberately asymmetric — a lack of evidence never *proves* the email is ham. This is **hypothesis testing**, the topic of this lesson, and the engine behind A/B testing at the lesson's end. It is based on the first video of Lesson 8 (Hypothesis Testing).

## 1. Introduction

Confidence intervals answered "what values are plausible for this parameter?" Hypothesis testing answers a sharper question: *is this specific belief about the population likely true or false?* This first notebook sets up the vocabulary and — most importantly — the asymmetric logic that everything later builds on.

After completing this notebook you will be able to:

- Define the null hypothesis $H_0$ (the baseline) and the alternative $H_1$ (the claim to establish), and design good pairs for real scenarios.
- Explain why the pair must be mutually exclusive with a true/false answer.
- State the asymmetry precisely: strong evidence rejects $H_0$; weak evidence only *fails to reject* it — it never proves it.
- Quantify "evidence against $H_0$" informally as *how unlikely the observation would be if $H_0$ were true*.
- Build a toy spam detector and watch both the logic and its two failure modes in action.

## 2. Theory

### The spam detector

An email arrives; it is either **ham** (not spam) or **spam** — and the detector must decide. By default, assume every email is ham. Why that default? Because the two mistakes are not equally bad: **deleting a good email is much worse than letting a spam email into the inbox.** The cautious assumption wins the role of baseline.

### The two hypotheses

- **Null hypothesis $H_0$** — the base assumption, the state where nothing special is happening: *this email is ham.*
- **Alternative hypothesis $H_1$** — the special claim you are trying to identify: *this email is spam.*

Two requirements make a well-designed pair:

1. **Mutually exclusive** — an email cannot be ham and spam at the same time.
2. **A true/false answer** — the question must be decidable, not a matter of degree.

Designing a good pair of hypotheses is half the craft: $H_0$ is the baseline, and $H_1$ — because of the asymmetry below — is usually *the statement you actually care about proving*.

### The asymmetry (the heart of the matter)

Evidence is gathered — sender, attachments, size, keywords — and weighed *against $H_0$*:

- **Plenty of evidence against $H_0$** → **reject $H_0$**, and accept $H_1$: the email goes to the spam folder.
- **Not enough evidence** → you **cannot reject $H_0$** — but this does **not** mean the email is ham! It means only that *you lack the evidence to show it's spam*.

The logic never proves $H_0$; it only fails to knock it down. (Courts run on the same asymmetry: a verdict of "not guilty" is a failure to prove guilt, not a certification of innocence.)

### What counts as evidence

For the spam decision: anything observable — the sender, the attachments, the email's size, specific keywords. Suppose an email contains *earn extra cash, risk free, dear friend, act immediately, apply now, winner*. These trigger phrases are **very unlikely to appear if the email is ham** — that is, unlikely *under $H_0$*. Observing several of them at once makes the email-as-ham story untenable: reject $H_0$, off to spam.

That phrase — "how likely is this observation *under the null hypothesis*" — is the informal seed of everything the coming videos formalize.

## 3. Worked Examples

### Example 1: designing hypothesis pairs

| scenario | $H_0$ (baseline) | $H_1$ (claim to establish) |
|---|---|---|
| spam filter | email is ham | email is spam |
| new drug trial | drug has no effect | drug has an effect |
| suspicious coin | coin is fair ($p = 0.5$) | coin is biased ($p \ne 0.5$) |
| website redesign (A/B) | new design changes nothing | new design changes conversions |

Note the pattern: $H_0$ is always "nothing is happening," and the interesting discovery is always parked in $H_1$ — because only $H_1$ can ever be *accepted*.

### Example 2: the four sentences

Only two decisions exist, and each licenses exactly one conclusion:

- reject $H_0$ → ✅ "the email is spam" (accept $H_1$)
- fail to reject $H_0$ → ✅ "there isn't enough evidence to call it spam"
- fail to reject $H_0$ → ❌ "the email is proven ham" — *never a valid conclusion*
- reject $H_0$ → ❌ "$H_1$ needed no evidence" — rejection is *earned* by evidence, never assumed

### Example 3: evidence, informally quantified

Say a given trigger phrase appears in about 0.1% of ham emails, independently of the others. One email containing **three** such phrases has probability roughly $0.001^3 = 10^{-9}$ *if it is ham*. You now hold an observation that is absurdly unlikely under $H_0$ — so believing $H_0$ requires believing a one-in-a-billion coincidence. Rejecting it is the only reasonable move. (How unlikely is unlikely *enough*? That threshold gets a name and a formal treatment in the next videos.)

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project), then a toy spam world: trigger phrases appear rarely in ham, frequently in spam.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(8)

TRIGGERS = ["earn extra cash", "risk free", "dear friend",
            "act immediately", "apply now", "winner"]
P_TRIGGER_HAM = 0.001      # each phrase appears in 0.1% of ham emails
P_TRIGGER_SPAM = 0.40      # ...and in 40% of spam emails

def p_evidence_given_ham(k):
    """P(this many trigger phrases | H0: ham), phrases independent."""
    return P_TRIGGER_HAM**k * (1 - P_TRIGGER_HAM)**(len(TRIGGERS) - k)

### The video's email, weighed against $H_0$

In [ ]:
email_phrases = ["earn extra cash", "risk free", "dear friend",
                 "act immediately", "apply now", "winner"]
k = len(email_phrases)
print(f"trigger phrases found: {k} of {len(TRIGGERS)}")
print(f"P(seeing this | H0: ham) = {p_evidence_given_ham(k):.2e}")
print(f"P(seeing this | H1: spam) = {P_TRIGGER_SPAM**k * (1 - P_TRIGGER_SPAM)**0:.4f}")
print("\nUnder H0 this email is a once-in-10^18 coincidence; under H1 it is")
print("perfectly ordinary. Verdict: reject H0 -- off to the spam folder.")

### A day of email, decided

Simulate 10,000 emails (95% ham), count each one's trigger phrases, and apply a simple rule: reject $H_0$ when the count is 2 or more. Watch the asymmetry in the output language: the inbox is full of "not shown to be spam," not "proven ham".

In [ ]:
n_emails = 10_000
is_spam = rng.random(n_emails) < 0.05
trigger_counts = np.where(
    is_spam,
    rng.binomial(len(TRIGGERS), P_TRIGGER_SPAM, size=n_emails),
    rng.binomial(len(TRIGGERS), P_TRIGGER_HAM, size=n_emails))

reject_h0 = trigger_counts >= 2                      # the decision rule

print(f"emails: {n_emails} ({(~is_spam).sum()} ham, {is_spam.sum()} spam)")
print(f"sent to spam folder (H0 rejected): {reject_h0.sum()}")
print(f"kept in inbox (H0 not rejected):   {(~reject_h0).sum()}")
print()
good_flagged = (reject_h0 & ~is_spam).sum()
spam_missed = (~reject_h0 & is_spam).sum()
print(f"mistake type 1 -- good email flagged as spam: {good_flagged} "
      f"({good_flagged / (~is_spam).sum():.2%} of ham)")
print(f"mistake type 2 -- spam left in the inbox:     {spam_missed} "
      f"({spam_missed / is_spam.sum():.2%} of spam)")
print("\n(These two mistakes get names and a formal treatment very soon.)")

## 5. Visualizations

### Why trigger phrases are evidence

The same observable — a phrase count — has wildly different distributions under the two hypotheses. Evidence is exactly this separation.

In [ ]:
counts = np.arange(0, 7)
from math import comb
pmf_ham = [comb(6, int(c)) * P_TRIGGER_HAM**c * (1 - P_TRIGGER_HAM)**(6 - c)
           for c in counts]
pmf_spam = [comb(6, int(c)) * P_TRIGGER_SPAM**c * (1 - P_TRIGGER_SPAM)**(6 - c)
            for c in counts]

plt.figure(figsize=(9.5, 4.6))
plt.bar(counts - 0.2, pmf_ham, 0.4, color="tab:green", label="under $H_0$: ham")
plt.bar(counts + 0.2, pmf_spam, 0.4, color="tab:red", label="under $H_1$: spam")
plt.axvline(1.5, color="black", linestyle="--", linewidth=1.2,
            label="decision rule: reject $H_0$ at 2+")
plt.yscale("log")
plt.ylim(1e-19, 3)
plt.xlabel("trigger phrases in the email")
plt.ylabel("probability (log scale)")
plt.title("One observable, two stories: counts that are ordinary for spam are astronomical for ham")
plt.grid(True, alpha=0.3, axis="y")
plt.legend()
plt.show()

### The decision, drawn as a courtroom

Every email starts at the presumption of ham; only the weight of evidence can move it. Both decision outcomes and both mistake types in one picture.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 5))
jitter_x = rng.normal(0, 0.07, size=n_emails)
jitter_y = rng.normal(0, 0.11, size=n_emails)
colors = np.where(is_spam, "tab:red", "tab:green")
ax.scatter(trigger_counts + jitter_x, jitter_y + is_spam.astype(int),
           s=6, c=colors, alpha=0.35)
ax.axvline(1.5, color="black", linestyle="--", linewidth=1.5)
ax.set_yticks([0, 1])
ax.set_yticklabels(["truly ham\n(green)", "truly spam\n(red)"])
ax.set_xlabel("trigger phrases counted (the evidence)")
ax.annotate("fail to reject $H_0$\n(inbox: 'not shown to be spam')",
            xy=(0.4, 0.5), ha="center", fontsize=10)
ax.annotate("reject $H_0$\n(spam folder)", xy=(3.6, 0.5), ha="center", fontsize=10)
ax.annotate("spam that slipped by\n(evidence too weak)", xy=(0.75, 1.30),
            ha="center", fontsize=8, color="tab:red")
ax.annotate("ham wrongly flagged would land here\n"
            f"({(reject_h0 & ~is_spam).sum()} in this batch -- rare by design)",
            xy=(3.6, -0.35), ha="center", fontsize=8, color="tab:green")
ax.set_xlim(-0.5, 6.5)
ax.set_ylim(-0.55, 1.6)
ax.set_title("The asymmetric verdicts: evidence can convict (spam), never acquit (ham)")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

### The whole story on one page

Everything above in a single sheet: the trigger-phrase counts under H₀ and H₁, how unlikely the evidence is under H₀ (10⁻¹⁸ for the video's email), the simulated day of 10,000 emails under the 2+ rule, and the valid and invalid conclusions.

<img src="img/01_intro_to_hypothesis_testing.png" alt="Concept sheet for this notebook" width="900">

## 6. Exercises

### Basic

**Exercise 1.** Write $H_0$ and $H_1$ for each: (a) testing whether a die is loaded; (b) testing whether a new checkout page increases sales; (c) a medical test for a rare disease. Check each pair for mutual exclusivity and a true/false answer.

**Exercise 2.** Your detector examines an email and finds zero trigger phrases. Which statements are valid conclusions? (a) "the email is ham"; (b) "$H_0$ was not rejected"; (c) "there is not enough evidence that the email is spam"; (d) "$H_0$ is true."

### Intermediate

**Exercise 3.** With $P(\text{phrase} \mid \text{ham}) = 0.001$ per phrase (independent, 6 phrases total), compute $P(\text{at least } k \text{ phrases} \mid H_0)$ for $k = 1, 2, 3$. At what $k$ does the evidence pass *your* personal threshold of "too unlikely to believe $H_0$," and what number did you implicitly choose as that threshold?

**Exercise 4.** *Swapping the default.* Suppose the detector took $H_0$: "email is spam" instead. Rewrite the two verdicts and their valid conclusions, and explain — using the relative cost of the two mistakes — why this design is worse for an email inbox but might be *right* for, say, screening luggage for explosives.

### Advanced

**Exercise 5.** *Tuning the rule.* Sweep the decision threshold from "1+ phrases" to "4+ phrases" on a fresh simulation. Tabulate both mistake rates at each threshold and describe the trade-off. Which threshold would you ship, given that a flagged good email costs you 100× more than a missed spam?

**Exercise 6.** *The base-rate reunion.* With 95% ham and the rule "reject at 2+," use Bayes' theorem (Lesson 1!) to compute the probability that a *flagged* email is actually ham. Verify against the simulation. Why does this number stay small here, and what would happen to it if spam were rarer (0.1% of mail)?

## 7. Solutions

In [ ]:
# Exercise 1: hypothesis pairs
print("(a) H0: the die is fair (all faces 1/6).  H1: the die is loaded.")
print("(b) H0: the new page changes nothing.     H1: the new page changes sales.")
print("(c) H0: the patient does not have the disease.  H1: the patient has it.")
print("Each pair: mutually exclusive, and a definite true/false fact of the world.")

In [ ]:
# Exercise 2: valid conclusions at zero evidence
print("(b) and (c) are valid -- they describe the decision and the evidence.")
print("(a) and (d) are NOT: failing to reject H0 never establishes H0. The email")
print("may well be a cleverly written spam with no trigger phrases at all.")

In [ ]:
# Exercise 3: tail probabilities under H0
from math import comb
for k in (1, 2, 3):
    p_at_least = sum(comb(6, j) * P_TRIGGER_HAM**j * (1 - P_TRIGGER_HAM)**(6 - j)
                     for j in range(k, 7))
    print(f"P(>= {k} phrases | ham) = {p_at_least:.2e}")
print("\nAlready k = 1 is a 1-in-167 event under H0; k = 2 is ~1 in 67,000.")
print("Whatever k you picked, you implicitly chose a smallness threshold for")
print("'probability of the evidence under H0' -- hold that thought.")

In [ ]:
# Exercise 4: swapping the default
print("With H0: spam --")
print("  reject H0        -> conclude 'ham', deliver the email.")
print("  fail to reject   -> 'not enough evidence of ham' -> delete it.")
print("Now the DEFAULT action destroys mail: every ambiguous email dies unless it")
print("proves innocence. Bad for inboxes, where deleting good mail is the costly")
print("error. Right for luggage screening: there the costly error is the miss,")
print("so the safe default must be 'suspicious until shown harmless'.")
print("Rule: H0 gets the position whose wrongful REJECTION is the expensive mistake.")

In [ ]:
# Exercise 5: threshold sweep
rng5 = np.random.default_rng(99)
is_spam5 = rng5.random(200_000) < 0.05
counts5 = np.where(is_spam5,
                   rng5.binomial(6, P_TRIGGER_SPAM, 200_000),
                   rng5.binomial(6, P_TRIGGER_HAM, 200_000))

print(f"{'rule':>10s} {'ham flagged':>12s} {'spam missed':>12s} {'cost (100:1)':>13s}")
for thr in (1, 2, 3, 4):
    flag = counts5 >= thr
    ham_flagged = (flag & ~is_spam5).mean() / (~is_spam5).mean()
    spam_missed = (~flag & is_spam5).mean() / is_spam5.mean()
    cost = 100 * ham_flagged + spam_missed
    print(f"{'>= ' + str(thr):>10s} {ham_flagged:>12.3%} {spam_missed:>12.2%} {cost:>13.3f}")
print("\nRaising the bar trades missed spam for protected ham. At 100:1 costs the")
print("sweep favors '>= 2': phrase-count 1 still flags too much ham at '>= 1',")
print("while '>= 3' starts missing spam for nearly no ham savings.")

In [ ]:
# Exercise 6: Bayes on the flagged pile
from math import comb
p_flag_ham = sum(comb(6, j) * P_TRIGGER_HAM**j * (1 - P_TRIGGER_HAM)**(6 - j)
                 for j in range(2, 7))
p_flag_spam = sum(comb(6, j) * P_TRIGGER_SPAM**j * (1 - P_TRIGGER_SPAM)**(6 - j)
                  for j in range(2, 7))
prior_ham, prior_spam = 0.95, 0.05

posterior_ham = (p_flag_ham * prior_ham /
                 (p_flag_ham * prior_ham + p_flag_spam * prior_spam))
print(f"P(ham | flagged) = {posterior_ham:.2e}   (Bayes)")

flagged = trigger_counts >= 2
print(f"simulation: {(flagged & ~is_spam).sum()} of {flagged.sum()} flagged "
      f"emails were ham = {(flagged & ~is_spam).sum() / flagged.sum():.2e}")

prior_spam_rare = 0.001
posterior_ham_rare = (p_flag_ham * (1 - prior_spam_rare) /
                      (p_flag_ham * (1 - prior_spam_rare) + p_flag_spam * prior_spam_rare))
print(f"\nif spam were 0.1% of mail: P(ham | flagged) = {posterior_ham_rare:.1%}")
print("Rare targets poison even good detectors with false alarms -- the Lesson 1")
print("disease-test lesson, back again in the spam folder.")

## 8. Conclusion

- **Hypothesis testing** decides between two competing claims about the world using data: the **null hypothesis $H_0$** (the safe baseline — *ham*, *no effect*, *fair coin*) and the **alternative $H_1$** (the claim you want to establish — *spam*, *the drug works*).
- A good pair is **mutually exclusive** with a **true/false answer** — and the baseline goes to the side whose wrongful rejection is the costlier mistake (deleting good email beats reading spam).
- The logic is **asymmetric**: strong evidence against $H_0$ → reject it and accept $H_1$; weak evidence → *fail to reject* — which never, ever proves $H_0$.
- Evidence means observations that are **unlikely under $H_0$**: six trigger phrases in one email is a $10^{-18}$ coincidence if the email is ham, so ham loses.
- Lurking in the toy detector: two distinct mistakes (good email flagged, spam missed), a tunable threshold trading one against the other, and a base-rate effect on the flagged pile — all of which the coming videos formalize.

Next: putting numbers on "unlikely enough" — and names on the two ways a test can be wrong.